# ALL — Object-Oriented Programming in Java (Consolidated)

A complete, master-level reference: the four pillars, class mechanics, interfaces, generics, records, sealed types, SOLID, and the essential design patterns — each with runnable demos and real-world tasks.

> **For experienced devs:** sections are collapsible. Skim theory, run demos, then grind the task cells.

> **Kernel tip:** define classes in a cell, then use top-level statements to exercise them (no `main` needed).

# Overview — The 4 Pillars

## What OOP gives you

```mermaid
mindmap
  root((OOP))
    Encapsulation
      hide state
      expose behavior
    Inheritance
      is-a
      reuse
    Polymorphism
      overriding
      overloading
    Abstraction
      interfaces
      abstract classes
```

| Pillar | One-liner | Java mechanism |
|--------|-----------|----------------|
| **Encapsulation** | Hide data, expose intent | `private` fields + methods, records |
| **Inheritance** | Reuse via *is-a* | `extends`, `super` |
| **Polymorphism** | One interface, many forms | overriding (runtime), overloading (compile) |
| **Abstraction** | Program to a contract | `interface`, `abstract class` |

**Design compass:** favor composition over inheritance, program to interfaces, keep classes cohesive and loosely coupled (SOLID).

##### Tasks — Warm-up / Mental model

> Mix of core mechanics · design · real-world backend scenarios.

1. Write a one-paragraph comment mapping each pillar to a class in a system you've built (e.g., an order service).
2. Model a `BankAccount`: private balance, `deposit`/`withdraw` with validation, no direct field access — name which pillar each choice demonstrates.
3. **Real-world** — Sketch (as classes/interfaces in comments) a payment module showing all 4 pillars: `PaymentMethod` interface, `Card`/`UPI` implementations, an abstract `PaymentProcessor`, encapsulated credentials.


# Classes, Objects & Initialization

## Anatomy & initialization order

- **Fields** (instance vs `static`), **methods**, **constructors**, `this`, `static`/instance initializer blocks.
- **Initialization order** when `new` runs:
  1. Static fields + static blocks (once per class load, top→bottom)
  2. Instance fields + instance initializer blocks (each `new`, top→bottom)
  3. Constructor body (after `super(...)`)

```mermaid
flowchart TB
    A[Class load] --> B[static fields and static blocks]
    B --> C[new called]
    C --> D[super constructor]
    D --> E[instance fields and init blocks]
    E --> F[constructor body]
```

| Concept | Notes |
|---------|-------|
| `static` member | Belongs to class, shared; access via `ClassName.member` |
| Instance member | Per-object copy |
| `this()` | Call another constructor (must be first statement) |
| `super()` | Call parent constructor (implicit if omitted) |
| Static block | One-time class setup (e.g., load table) |
| Instance block | Runs before every constructor body |

In [ ]:
// Demo: initialization order
class Widget {
    static int created;
    static { System.out.println("static block (once)"); }
    int id;
    { id = ++created; System.out.println("instance block -> id=" + id); }
    Widget() { System.out.println("ctor body id=" + id); }
}
new Widget();
new Widget();
System.out.println("total created = " + Widget.created);


##### Tasks — Classes & Initialization

> Mix of core mechanics · design · real-world backend scenarios.

1. Create a `Counter` with a `static` total across all instances and a per-instance count; prove the difference.
2. Use `this(...)` constructor chaining: `Rectangle()`, `Rectangle(side)`, `Rectangle(w,h)` all funnel to one canonical constructor.
3. Add a `static` factory `of(...)` and make the constructor `private`; discuss why (control instantiation, caching).
4. Predict then verify the exact print order for a class with static block, instance block, and two constructors.
5. **Real-world** — Build a `IdGenerator` singleton-ish `static` sequence used by an `Entity` base so every new entity gets a unique id.


# Encapsulation & Immutability

## Access control + immutable objects

| Modifier | Same class | Same package | Subclass | World |
|----------|:----------:|:------------:|:--------:|:-----:|
| `private` | ✅ | ❌ | ❌ | ❌ |
| *(default)* | ✅ | ✅ | ❌ | ❌ |
| `protected` | ✅ | ✅ | ✅ | ❌ |
| `public` | ✅ | ✅ | ✅ | ✅ |

**Immutability recipe:** `final` class, `private final` fields, no setters, defensive copies of mutable inputs/outputs, initialize all in constructor. Immutable objects are inherently thread-safe and safe as map keys.

`record` (Java 16+) gives you an immutable data carrier with auto `equals`/`hashCode`/`toString`/accessors.

In [ ]:
// Demo: immutable value object with defensive copy
import java.util.*;
final class Money {
    private final long cents;
    private final String currency;
    Money(long cents, String currency) { this.cents = cents; this.currency = currency; }
    long cents() { return cents; }
    String currency() { return currency; }
    Money plus(Money o) {
        if (!currency.equals(o.currency)) throw new IllegalArgumentException("currency mismatch");
        return new Money(cents + o.cents, currency);
    }
    public String toString() { return String.format("%.2f %s", cents/100.0, currency); }
}
Money a = new Money(1050, "USD");
System.out.println(a.plus(new Money(295, "USD")));


##### Tasks — Encapsulation & Immutability

> Mix of core mechanics · design · real-world backend scenarios.

1. Convert a mutable `Point{x,y}` (public fields) into an encapsulated, then a fully **immutable** version; add `withX(int)` returning a copy.
2. Build an immutable `Range(lo, hi)` with validation and `contains`, `overlaps`, `union` returning new instances.
3. Demonstrate a defensive-copy bug: store a `Date`/`int[]` without copying, mutate the caller's reference, and show the object changed; then fix it.
4. Convert an immutable class to a `record`; note what boilerplate disappears and add a **compact constructor** for validation.
5. **Real-world** — Immutable `HttpRequest(method, url, headers, body)` with a builder; headers stored as an unmodifiable copy.
6. **Big task** — Implement a builder that produces a deeply immutable `Config` (nested immutable maps/lists) and reject nulls; prove it's safe to share across threads.


# Inheritance

## extends, super, overriding, final

- `extends` a single class (Java has **single** class inheritance); multiple **interface** inheritance is allowed.
- `super.method()` / `super(...)` reach the parent.
- **Overriding** rules: same signature, return type covariant, access not more restrictive, cannot throw broader checked exceptions; use `@Override`.
- `final` class → cannot extend; `final` method → cannot override.
- Every class implicitly extends `Object`.

```mermaid
classDiagram
    Animal <|-- Dog
    Animal <|-- Cat
    class Animal { +String name +speak() }
    class Dog { +speak() }
    class Cat { +speak() }
```

**Liskov reminder:** a subclass must be usable anywhere its superclass is expected — don't strengthen preconditions or weaken postconditions.

In [ ]:
// Demo: overriding + dynamic dispatch + super
class Animal {
    protected String name;
    Animal(String name) { this.name = name; }
    String speak() { return name + " makes a sound"; }
}
class Dog extends Animal {
    Dog(String name) { super(name); }
    @Override String speak() { return name + " barks"; }
}
Animal a = new Dog("Rex");   // upcast
System.out.println(a.speak()); // dynamic dispatch -> Dog.speak


##### Tasks — Inheritance

> Mix of core mechanics · design · real-world backend scenarios.

1. Build `Shape` → `Circle`/`Rectangle`/`Triangle` with overridden `area()`/`perimeter()`; store them in a `List<Shape>` and total the area.
2. Show constructor chaining across 3 levels (`A`→`B`→`C`) printing which `super()` runs when.
3. Demonstrate a covariant return type: parent `copy()` returns `Animal`, child returns `Dog`.
4. Break the Liskov Substitution Principle with the classic `Square extends Rectangle` setter trap; then redesign to fix it.
5. **Real-world** — Model a class hierarchy for notifications: abstract `Notification` with `send()`, subclasses `Email`, `SMS`, `Push`; a dispatcher iterates polymorphically.
6. **Big task** — Design a plugin-style `AbstractRepository<T,ID>` with template CRUD and a `JdbcRepository`/`InMemoryRepository` subclass; only the storage differs.


# Polymorphism

## Compile-time (overloading) vs runtime (overriding)

| | Overloading | Overriding |
|---|-------------|------------|
| Binding | Compile-time (static) | Runtime (dynamic) |
| Signature | Different params | Same signature |
| Where | Same class | Subclass |
| Return type | Can differ | Covariant only |

- **Dynamic dispatch:** the actual object's method runs, not the reference type's.
- **Upcasting** is implicit and safe; **downcasting** needs a check (`instanceof`, pattern matching in Java 16+).
- Fields are **not** polymorphic (resolved by reference type); `static` methods are hidden, not overridden.

In [ ]:
// Demo: overloading + pattern-matching instanceof
static String describe(Object o) {
    if (o instanceof Integer i) return "int " + (i * 2);
    if (o instanceof String s) return "str len " + s.length();
    return "unknown";
}
String area(int s) { return "square " + (s*s); }        // overload 1
String area(int w, int h) { return "rect " + (w*h); }    // overload 2
System.out.println(describe(21));
System.out.println(describe("hello"));
System.out.println(area(5));
System.out.println(area(3, 4));


##### Tasks — Polymorphism

> Mix of core mechanics · design · real-world backend scenarios.

1. Write an overloaded `max` for `int`, `double`, and `Comparable<T>`; show which the compiler picks.
2. Prove fields are not polymorphic: same-named field in parent & child, access via parent reference.
3. Prove `static` methods are hidden not overridden (call via parent vs child reference).
4. Refactor a big `if/else` type-switch into polymorphic method dispatch (replace conditional with polymorphism).
5. Use Java 21 **switch pattern matching** to handle a sealed `Shape` hierarchy exhaustively.
6. **Real-world** — A `ReportExporter` that takes `List<Exportable>` and calls `export()` polymorphically for CSV/PDF/JSON exporters.


# Abstraction — Abstract Classes vs Interfaces

## When to use which

| | Abstract class | Interface |
|---|----------------|-----------|
| State (fields) | Yes | Only `static final` constants |
| Constructors | Yes | No |
| Multiple inheritance | No (single) | Yes (many) |
| Methods | abstract + concrete | abstract + `default` + `static` + `private` (Java 9+) |
| Use when | Shared state/base impl, *is-a* | Capability/contract, mixin |

- **Default methods** let interfaces evolve without breaking implementers.
- **Diamond problem:** if two interfaces provide the same default, the implementer must override and can call `Iface.super.method()`.
- **Functional interface:** exactly one abstract method → usable as a lambda (`@FunctionalInterface`).

In [ ]:
// Demo: interface with default + diamond resolution
interface Walker { default String move() { return "walk"; } }
interface Swimmer { default String move() { return "swim"; } }
class Amphibian implements Walker, Swimmer {
    public String move() { return Walker.super.move() + "+" + Swimmer.super.move(); }
}
System.out.println(new Amphibian().move());

@FunctionalInterface interface Op { int apply(int a, int b); }
Op add = (x, y) -> x + y;
System.out.println("add: " + add.apply(3, 4));


##### Tasks — Abstraction

> Mix of core mechanics · design · real-world backend scenarios.

1. Design `abstract class Vehicle` (shared fuel/state + template `start()`) and interface `Electric` (capability); a `Tesla` uses both.
2. Create an interface with a `default` method and evolve it by adding another default — show existing implementers still compile.
3. Trigger and resolve the diamond problem with two conflicting default methods.
4. Write three functional interfaces and pass lambdas; then rewrite using the JDK equivalents (`Function`, `BiFunction`, `Predicate`, `Supplier`, `Consumer`).
5. Add a `private` interface helper method (Java 9+) shared by two default methods.
6. **Real-world** — Define a `Cache<K,V>` interface (get/put/evict) with a `default getOrLoad(key, loader)`; provide an in-memory implementation.
7. **Big task** — Design a `Pipeline<T>` abstraction: chain `Stage<T>` steps (interface), support `addStage`, run in order, and short-circuit on failure.


# The Object Class — equals, hashCode, toString, clone

## Contracts you must respect

**equals/hashCode contract:**
- `equals` reflexive, symmetric, transitive, consistent; `x.equals(null)` is false.
- If `a.equals(b)` then `a.hashCode() == b.hashCode()` (the reverse need not hold).
- Break this → broken `HashMap`/`HashSet` behavior.

| Method | Purpose | Notes |
|--------|---------|-------|
| `equals(Object)` | Logical equality | Override with `hashCode` together |
| `hashCode()` | Bucket for hashing | Use `Objects.hash(...)` |
| `toString()` | Debug/log representation | Always helpful |
| `clone()` | Copy | Prefer copy constructors/factories; shallow by default |
| `getClass()` | Runtime type | Use in `equals` (or `instanceof`) |

Prefer `Objects.equals`, `Objects.hash`, and copy constructors over `Cloneable`.

In [ ]:
// Demo: correct equals/hashCode
import java.util.*;
final class Point {
    final int x, y;
    Point(int x, int y) { this.x = x; this.y = y; }
    public boolean equals(Object o) {
        if (this == o) return true;
        if (!(o instanceof Point p)) return false;
        return x == p.x && y == p.y;
    }
    public int hashCode() { return Objects.hash(x, y); }
    public String toString() { return "(" + x + "," + y + ")"; }
}
Set<Point> set = new HashSet<>();
set.add(new Point(1,2)); set.add(new Point(1,2));
System.out.println("size (should be 1): " + set.size());


##### Tasks — Object Class

> Mix of core mechanics · design · real-world backend scenarios.

1. Implement `equals`/`hashCode`/`toString` for an `Employee(id, name, dept)`; put in a `HashSet` and prove dedup.
2. Break the contract (equal objects, different hashCodes) and demonstrate a lookup miss in a `HashMap`.
3. Write a **shallow** vs **deep** copy for a class holding a `List`; mutate and show the difference.
4. Replace `clone()` with a copy constructor and a static `copyOf` factory.
5. Show why using mutable fields in `equals`/`hashCode` corrupts a `HashSet` after mutation.
6. **Real-world** — Implement `equals`/`hashCode` for a composite DB key `(tenantId, entityId)` used as a map key in a cache.


# Composition vs Inheritance & Object Relationships

## Favor composition; know the relationships

```mermaid
classDiagram
    Car *-- Engine : composition (owns, lifecycle)
    Team o-- Player : aggregation (has, independent)
    Order --> PricingService : dependency (uses)
    Bird ..|> Flyer : realization (implements)
```

| Relationship | Meaning | Symbol | Example |
|--------------|---------|--------|---------|
| Composition | strong *has-a*, owned lifecycle | ◆ filled | `Car`–`Engine` |
| Aggregation | weak *has-a*, independent | ◇ open | `Team`–`Player` |
| Association | uses / linked | → | `Student`–`Course` |
| Dependency | transient use (param/local) | ⇢ | `Order`–`PricingService` |
| Inheritance | *is-a* | ▷ | `Dog`–`Animal` |

**Rule of thumb:** if you're only reusing code (not modeling *is-a*), compose + delegate instead of extending.

In [ ]:
// Demo: composition + delegation
class Engine { String start() { return "vroom"; } }
class Car {
    private final Engine engine = new Engine();  // composition
    String drive() { return "car: " + engine.start(); } // delegation
}
System.out.println(new Car().drive());


##### Tasks — Composition & Relationships

> Mix of core mechanics · design · real-world backend scenarios.

1. Refactor a `Stack extends Vector` design into `Stack` that **composes** a `List` (hide unwanted inherited methods).
2. Model `Car`(composition of `Engine`,`Wheel[]`) and `Team`(aggregation of `Player`s); show lifecycle differences.
3. Implement the **delegation** pattern: `LoggingList<T>` wraps any `List<T>` and logs mutations (forwarding methods).
4. Identify each relationship type in a mini e-commerce model (Order, LineItem, Product, Customer, PricingService).
5. **Real-world** — Build a `RetryingClient` that composes an `HttpClient` interface and adds retry/backoff via delegation, no inheritance.
6. **Big task** — Reproduce the JDK's `Collections.unmodifiableList` idea: a forwarding wrapper that throws on mutations.


# Nested, Inner, Local & Anonymous Classes

## Four kinds of nested classes

| Kind | Holds outer ref? | Use |
|------|:----------------:|-----|
| `static` nested | No | Helper tied to outer namespace (e.g., `Map.Entry`) |
| Inner (non-static) | Yes | Needs outer instance state (iterators) |
| Local (in a method) | Yes | One-off within a method |
| Anonymous | Yes | Inline one-shot implementation of an interface/class |

- Inner classes capture the enclosing instance (`Outer.this`).
- Local/anonymous classes capture **effectively final** locals.
- Lambdas are lighter than anonymous classes and don't create a new scope for `this`.

In [ ]:
// Demo: static nested vs inner vs anonymous
class Outer {
    private int val = 42;
    static class Nested { String hi() { return "static nested"; } }
    class Inner { String hi() { return "inner sees val=" + val; } }
}
Outer o = new Outer();
System.out.println(new Outer.Nested().hi());
System.out.println(o.new Inner().hi());
Runnable r = new Runnable() { public void run() { System.out.println("anonymous run"); } };
r.run();


##### Tasks — Nested & Anonymous Classes

> Mix of core mechanics · design · real-world backend scenarios.

1. Implement a linked list where the `Node` is a `private static` nested class.
2. Write an inner-class `Iterator` for a custom collection that accesses the outer instance's array.
3. Replace an anonymous `Comparator` with a lambda and then a method reference — three ways to sort.
4. Show a local class defined inside a method that captures an effectively-final variable.
5. **Real-world** — Build a fluent event `Subscription` where `subscribe(listener)` returns an anonymous `AutoCloseable` that unsubscribes.


# Enums — More Than Constants

## Enums with state, behavior, and per-constant overrides

- Enums are full classes: fields, constructors, methods.
- **Constant-specific method bodies** (each constant overrides an abstract method).
- `EnumSet` / `EnumMap` are compact, fast collections keyed by enum.
- The **enum singleton** is the safest singleton (serialization + reflection safe).

In [ ]:
// Demo: enum with fields + abstract method per constant
enum Op {
    ADD("+") { int apply(int a,int b){ return a+b; } },
    MUL("*") { int apply(int a,int b){ return a*b; } };
    private final String sym;
    Op(String s){ this.sym = s; }
    abstract int apply(int a, int b);
    String sym(){ return sym; }
}
for (Op op : Op.values())
    System.out.println("3 " + op.sym() + " 4 = " + op.apply(3,4));


##### Tasks — Enums

> Mix of core mechanics · design · real-world backend scenarios.

1. Model `Planet` enum with mass/radius fields and a `surfaceGravity()` method.
2. Implement a calculator using an `Operation` enum with constant-specific `apply`.
3. Build a state machine `TrafficLight` where each constant knows its `next()`.
4. Use `EnumMap<Day, List<Task>>` for a weekly scheduler; compare with `HashMap`.
5. Implement a thread-safe **singleton via enum** (e.g., a config registry) and explain why it beats double-checked locking.
6. **Real-world** — Represent HTTP status codes as an enum with category helpers (`isSuccess`, `isClientError`) and message.


# Generics — Type Safety & Variance

## Generic classes/methods, bounds, wildcards, erasure

- **Generic class/method:** `class Box<T>`, `<T> T firstOf(List<T>)`.
- **Bounded type:** `<T extends Comparable<T>>`.
- **Wildcards & PECS** — *Producer Extends, Consumer Super*:
  - `List<? extends T>` → read (produce) T
  - `List<? super T>` → write (consume) T
- **Type erasure:** generics are compile-time; no `new T()`, no `T[]` directly, no primitive type args, `instanceof List<String>` illegal.

```mermaid
flowchart LR
    P["? extends T (Producer)"] -->|read as T| R[safe get]
    Cn["? super T (Consumer)"] -->|write T| W[safe add]
```

In [ ]:
// Demo: generic method + PECS
import java.util.*;
static <T extends Comparable<T>> T maxOf(List<? extends T> xs) {
    T best = xs.get(0);
    for (T x : xs) if (x.compareTo(best) > 0) best = x;
    return best;
}
static <T> void copy(List<? extends T> src, List<? super T> dst) {
    for (T x : src) dst.add(x);
}
System.out.println(maxOf(List.of(3, 9, 2, 7)));
List<Number> dst = new ArrayList<>();
copy(List.of(1, 2, 3), dst);
System.out.println(dst);


##### Tasks — Generics

> Mix of core mechanics · design · real-world backend scenarios.

1. Implement a generic `Pair<A,B>` and `Triple<A,B,C>` with `equals`/`hashCode`/`toString`.
2. Write a generic `Stack<T>` and a generic `swap(List<T>, i, j)`.
3. Implement `merge(List<? extends T>, List<? extends T>, Comparator<? super T>)` — apply PECS correctly.
4. Demonstrate a type-erasure limitation (can't do `new T[]`); work around it with `Class<T>` or `Object[]`.
5. Build a generic `Result<T>` (success value or error) with `map`/`flatMap`/`orElse` — a mini Either/Optional.
6. **Real-world** — Generic `Repository<T, ID>` interface + typed `EventRepository`; add a generic `Specification<T>` filter.
7. **Big task** — Generic type-safe heterogeneous container (`put(Class<T>, T)`, `get(Class<T>)`) — Josh Bloch's typesafe hetero container pattern.


# Records & Sealed Classes (Java 16/17+)

## Modern data & closed hierarchies

- **`record`**: immutable data carrier — auto `equals`/`hashCode`/`toString`/accessors; add **compact constructors** for validation; can implement interfaces; no extends.
- **`sealed`** class/interface + `permits`: restrict who can implement/extend → exhaustive `switch` pattern matching, safer domain modeling.

```java
sealed interface Shape permits Circle, Rect {}
record Circle(double r) implements Shape {}
record Rect(double w, double h) implements Shape {}
```

In [ ]:
// Demo: sealed + record + exhaustive switch (Java 21)
sealed interface Shape permits Circle, Rect {}
record Circle(double r) implements Shape {}
record Rect(double w, double h) implements Shape {}

static double area(Shape s) {
    return switch (s) {
        case Circle c -> Math.PI * c.r() * c.r();
        case Rect r   -> r.w() * r.h();
    };
}
System.out.printf("circle=%.2f rect=%.2f%n", area(new Circle(2)), area(new Rect(3,4)));


##### Tasks — Records & Sealed Types

> Mix of core mechanics · design · real-world backend scenarios.

1. Convert a verbose immutable DTO into a `record`; add a compact constructor that validates and normalizes fields.
2. Add a static factory and an extra derived method to a record.
3. Model a `sealed interface Json permits JsonObject, JsonArray, JsonString, JsonNumber, JsonBool, JsonNull` and write an exhaustive pretty-printer via switch.
4. Model domain events as a sealed hierarchy and handle them exhaustively (compiler enforces completeness).
5. **Real-world** — Represent an API `Result` as `sealed interface Result<T> permits Ok, Err`; write a handler that can't forget a case.
6. **Big task** — Build a tiny expression AST as sealed records (`Num`, `Add`, `Mul`, `Neg`) and write an `eval` + a `pretty` printer using switch patterns.


# Exceptions, try-with-resources & AutoCloseable

## OOP of error handling

| Type | Checked? | Use |
|------|:--------:|-----|
| `Error` | No | JVM/fatal — don't catch |
| `RuntimeException` | No (unchecked) | Programming errors, invalid args/state |
| checked `Exception` | Yes | Recoverable, caller must handle/declare |

- Prefer **unchecked** for programming errors; **checked** for recoverable conditions the caller can act on.
- Create meaningful **custom exceptions** (extend `RuntimeException` or `Exception`); preserve cause (`new X(msg, cause)`).
- **try-with-resources** auto-closes anything implementing `AutoCloseable` (reverse order); suppressed exceptions are attached.
- Don't swallow exceptions; don't use exceptions for control flow.

In [ ]:
// Demo: custom exception + try-with-resources
class Resource implements AutoCloseable {
    final String name;
    Resource(String n){ name = n; System.out.println("open " + n); }
    public void close(){ System.out.println("close " + name); }
}
class DomainException extends RuntimeException {
    DomainException(String m, Throwable cause){ super(m, cause); }
}
try (Resource a = new Resource("A"); Resource b = new Resource("B")) {
    System.out.println("use A,B");
    throw new IllegalStateException("boom");
} catch (RuntimeException e) {
    System.out.println("caught: " + e.getMessage());
}


##### Tasks — Exceptions

> Mix of core mechanics · design · real-world backend scenarios.

1. Design an exception hierarchy for a service: base `ServiceException` → `NotFoundException`, `ValidationException`, `ConflictException`.
2. Write a method that wraps a low-level `IOException` in a domain exception preserving the cause; print the full chain.
3. Implement an `AutoCloseable` `Transaction` (begin/commit/rollback) usable in try-with-resources with rollback on failure.
4. Show suppressed exceptions: throw in both the body and `close()`, then read `getSuppressed()`.
5. **Real-world** — Build a `retry(times, backoff, Callable<T>)` helper that retries only on transient exceptions and rethrows others.
6. **Big task** — Implement a `Result<T>`/`Try<T>` monad to avoid try/catch sprawl: `of(supplier)`, `map`, `recover`, `getOrThrow`.


# SOLID Principles

## Five principles for maintainable OO

| Letter | Principle | Smell it fixes |
|--------|-----------|----------------|
| **S** | Single Responsibility | God classes, mixed concerns |
| **O** | Open/Closed | Editing existing code for every new case |
| **L** | Liskov Substitution | Subclasses that break the base contract |
| **I** | Interface Segregation | Fat interfaces forcing empty impls |
| **D** | Dependency Inversion | High-level code bound to concretions |

```mermaid
flowchart LR
    HL[High-level policy] --> AB[Abstraction]
    LL[Low-level detail] --> AB
```

**DIP essence:** depend on abstractions, inject dependencies (constructor injection).

In [ ]:
// Demo: DIP + OCP with strategy injection
interface Discount { double apply(double price); }
class NoDiscount implements Discount { public double apply(double p){ return p; } }
class PercentOff implements Discount {
    final double pct; PercentOff(double pct){ this.pct = pct; }
    public double apply(double p){ return p * (1 - pct); }
}
class Checkout {
    private final Discount discount;              // depends on abstraction
    Checkout(Discount d){ this.discount = d; }    // injected
    double total(double price){ return discount.apply(price); }
}
System.out.println(new Checkout(new PercentOff(0.2)).total(100));


##### Tasks — SOLID

> Mix of core mechanics · design · real-world backend scenarios.

1. **SRP** — Split a `UserService` that also does validation, persistence, and email into focused collaborators.
2. **OCP** — Replace a growing `switch(paymentType)` with a `PaymentStrategy` interface + implementations; add a new type without editing existing code.
3. **LSP** — Find and fix an LSP violation (e.g., `Ostrich extends Bird` with `fly()`); redesign with capability interfaces.
4. **ISP** — Break a fat `Machine{print,scan,fax}` interface into `Printer`, `Scanner`, `Fax`; a simple printer implements only one.
5. **DIP** — Refactor a class that `new`s its dependency to accept it via constructor injection; write a fake for testing.
6. **Big task** — Take a deliberately bad 150-line 'OrderManager' god class (write it first) and refactor it to satisfy all five principles; note each change.


# Design Patterns — Creational

## Singleton · Factory · Abstract Factory · Builder · Prototype

| Pattern | Intent |
|---------|--------|
| Singleton | One instance, global access (prefer enum / DI) |
| Factory Method | Defer instantiation to subclasses/methods |
| Abstract Factory | Families of related objects |
| Builder | Construct complex objects step by step |
| Prototype | Clone existing instances |

In [ ]:
// Demo: Builder
class Pizza {
    final String size; final boolean cheese, pepperoni;
    private Pizza(Builder b){ size=b.size; cheese=b.cheese; pepperoni=b.pepperoni; }
    static class Builder {
        String size = "M"; boolean cheese, pepperoni;
        Builder size(String s){ this.size=s; return this; }
        Builder cheese(){ this.cheese=true; return this; }
        Builder pepperoni(){ this.pepperoni=true; return this; }
        Pizza build(){ return new Pizza(this); }
    }
    public String toString(){ return size+" cheese="+cheese+" pep="+pepperoni; }
}
System.out.println(new Pizza.Builder().size("L").cheese().build());


##### Tasks — Creational Patterns

> Mix of core mechanics · design · real-world backend scenarios.

1. Implement a thread-safe Singleton three ways: eager, double-checked locking (`volatile`), and enum; discuss trade-offs.
2. Build a `ShapeFactory` (factory method) and an `AbstractFactory` for GUI widgets (`WinButton`/`MacButton`).
3. Write a fluent **Builder** for an immutable `HttpRequest` with required + optional fields and validation in `build()`.
4. Implement **Prototype** with a `copy()`/registry of preconfigured objects.
5. **Real-world** — Build a `ConnectionPool` singleton that hands out and returns pooled connections (bounded, blocking when empty).
6. **Big task** — Combine Builder + Factory: a `QueryBuilder` producing immutable `Query` objects, with a `QueryFactory` for common query templates.


# Design Patterns — Structural

## Adapter · Decorator · Composite · Proxy · Facade · Bridge · Flyweight

| Pattern | Intent |
|---------|--------|
| Adapter | Make incompatible interfaces work together |
| Decorator | Add behavior dynamically (wrap) |
| Composite | Tree of part-whole objects treated uniformly |
| Proxy | Stand-in controlling access (lazy, remote, protection) |
| Facade | Simplified entry point over a subsystem |
| Bridge | Decouple abstraction from implementation |
| Flyweight | Share fine-grained objects to save memory |

In [ ]:
// Demo: Decorator
interface Coffee { double cost(); String desc(); }
class Espresso implements Coffee { public double cost(){return 2;} public String desc(){return "espresso";} }
abstract class AddOn implements Coffee {
    protected final Coffee base; AddOn(Coffee b){ base=b; }
}
class Milk extends AddOn { Milk(Coffee b){super(b);} public double cost(){return base.cost()+0.5;} public String desc(){return base.desc()+"+milk";} }
Coffee c = new Milk(new Milk(new Espresso()));
System.out.println(c.desc()+" = "+c.cost());


##### Tasks — Structural Patterns

> Mix of core mechanics · design · real-world backend scenarios.

1. **Adapter** — Adapt a legacy `LegacyLogger.write(String)` to a modern `Logger.info/warn/error` interface.
2. **Decorator** — Recreate the `java.io` stream idea: wrap a `DataSource` with `Buffered`, `Encrypted`, `Compressed` decorators.
3. **Composite** — Model a filesystem (`File`/`Directory`) and compute total size recursively via a common `Node` interface.
4. **Proxy** — Implement a lazy-loading / caching proxy in front of an expensive `ImageLoader`.
5. **Facade** — Wrap a multi-step order subsystem (inventory, payment, shipping) behind `OrderFacade.placeOrder(...)`.
6. **Real-world / Big task** — Build a rate-limiting + retry + logging **proxy** around an interface using dynamic delegation (or `java.lang.reflect.Proxy`).


# Design Patterns — Behavioral

## Strategy · Observer · Template Method · State · Command · Iterator · Chain of Responsibility · Mediator

| Pattern | Intent |
|---------|--------|
| Strategy | Swap algorithms behind an interface |
| Observer | Publish/subscribe notifications |
| Template Method | Fixed skeleton, overridable steps |
| State | Behavior changes with internal state |
| Command | Encapsulate a request as an object (undo) |
| Iterator | Sequential access without exposing internals |
| Chain of Responsibility | Pass request along handlers |
| Mediator | Centralize complex interactions |

In [ ]:
// Demo: Observer
import java.util.*;
interface Observer { void update(String event); }
class Subject {
    private final List<Observer> obs = new ArrayList<>();
    void subscribe(Observer o){ obs.add(o); }
    void publish(String e){ for (Observer o: obs) o.update(e); }
}
Subject s = new Subject();
s.subscribe(e -> System.out.println("A got " + e));
s.subscribe(e -> System.out.println("B got " + e));
s.publish("order.created");


##### Tasks — Behavioral Patterns

> Mix of core mechanics · design · real-world backend scenarios.

1. **Strategy** — A `Sorter` that accepts a `SortStrategy` (bubble/quick/`Comparator`); swap at runtime.
2. **Observer** — Build a typed event bus with `subscribe`/`unsubscribe`/`publish` and multiple listeners.
3. **Template Method** — Abstract `DataImporter` with `read`→`transform`→`write` skeleton; CSV & JSON subclasses fill steps.
4. **State** — Model an order lifecycle (`New`→`Paid`→`Shipped`→`Delivered`) where each state permits only valid transitions.
5. **Command** — Implement an editor with `execute`/`undo` commands and an undo stack.
6. **Chain of Responsibility** — Build a request-handling middleware chain (auth → rate-limit → logging → handler).
7. **Real-world / Big task** — Combine Strategy + Observer + Command in a mini trading engine: pricing strategy, order commands with undo, and market-data observers.


# Capstone Projects

## Put it all together

These integrate multiple pillars + patterns. Aim for clean interfaces, immutability where sensible, SOLID, and tests via `assert`/prints.

##### Tasks — Capstone

> Mix of core mechanics · design · real-world backend scenarios.

1. **Parking Lot** — Vehicles (enum sizes), spot allocation strategy, ticketing, pricing; interfaces + polymorphism + strategy.
2. **In-memory ORM-lite** — `@Entity`-style records, a generic `Repository<T,ID>`, unit-of-work, and a query builder.
3. **Event-driven notification system** — Sealed event types, observer subscribers, strategy for delivery channel, retry with backoff.
4. **Plugin framework** — Discover and load `Plugin` implementations (ServiceLoader-style), lifecycle `init/start/stop`, isolation via interfaces.
5. **State-machine workflow engine** — Define states/transitions/guards; execute a workflow; make it extensible (OCP) and testable (DIP).
6. **Mini DI container** — Register bindings (`bind(Interface).to(Impl)`), resolve with constructor injection and singleton scope.


# Quick Reference Card

| Goal | Reach for |
|------|-----------|
| Hide state | `private` + methods, `record` |
| Share behavior + state | `abstract class` |
| Capability / multiple inheritance | `interface` (+ `default`) |
| Immutable data | `record` / final fields + defensive copies |
| Closed hierarchy + exhaustive switch | `sealed` + `permits` |
| Type safety over collections | generics + PECS wildcards |
| Swap behavior at runtime | Strategy |
| Add behavior without subclassing | Decorator |
| Notify many | Observer |
| Build complex objects | Builder |
| Depend on abstractions | DIP + constructor injection |

---

✅ Every pillar, modern feature, SOLID principle, and the essential GoF patterns are covered. Fill the task cells — build the capstones for real mastery.